In [60]:
# Import libraries
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import roc_auc_score

In [61]:
# Loading post-eda data 
notebook_path = Path.cwd()
project_path = notebook_path.parent
data_path = project_path / "data" / "processed" / "post_eda_master_table.parquet"

df = pd.read_parquet(data_path)


We pick up from eda.ipynb, which has already applied 3 corrections to the data:
1. dtype fixes ('seqn' → int, ordinal/categorical columns → 'Int64')

2. 'very_loud_noise': structural NULLs recoded to "no exposure" based on confirmed AUQ_I skip logic (AUQ350 was only asked to people who answered Yes to AUQ331)

3. Binary features ('sex', 'military_service', disease flags) relabeled from 1/2 to 0/1

From here, we handle the remaining missingness identified during EDA.

Before starting the cleaning, I need to drop the features that provide no predictive value. Because my continous and categorical data use different methods for associating with hearing loss (rank biserial - cramer's v), I can't use a single cut-off threshold for both and I don't want to just rely on my clinical knowledge to decide which features are not worth keeping. So I need one single method for calculating the predictive power of each feature against HL and I will use ROC-AUC for that.

In [62]:
# a function to calculate auc
def get_auc(df):
    results= []
    hl = df['hl_presence']

    features = [col for col in df.columns
            if col not in ['seqn', 'hl_presence', 'hl_severity']]
    
    for col in features:
        valid_idx = df[col].notna() & hl.notna()
        x_val = df.loc[valid_idx, col]
        y_val = hl.loc[valid_idx]
        auc = roc_auc_score(y_val, x_val)
        results.append({
            'Feature': col,
            'ROC_SCORE': max(auc, 1 - auc) # Because we could have a negative correlation (auc < 0.5), we do this so all of them are between 0.5 and 1.
        })

    results_df = pd.DataFrame(results)
    return(results_df.sort_values('ROC_SCORE', ascending= False))

print(get_auc(df))

                   Feature  ROC_SCORE
1                      age   0.798761
11                    lead   0.650825
10          blood_pressure   0.626034
3                education   0.597063
0                      sex   0.589845
17                diabetes   0.588884
7               loud_noise   0.570673
12                 cadmium   0.557254
8          very_loud_noise   0.555839
9                      bmi   0.545036
5   firearm_and_protection   0.543493
4     income_poverty_ratio   0.538640
6   firearm_exposure_level   0.537902
2         military_service   0.534377
15                  stroke   0.524122
13           heart_disease   0.522675
16                  cancer   0.522260
14            heart_attack   0.516134


I decide to drop every feature that has an AUC under 0.55.
But the problem is 2 of those columns which are  firearm_exposure_level and firearm_and_protection, can give us real insights about the probability of hearing loss as they can be a major factor in it. So I decided to create a new column named gun_risk and anyone with medium/high gun usage without protection will be classified as high risk and anybody else will be low risk (no or low gun usage or people who use protection > 50% of the time)

In [63]:
# high risk
high_risk =(df['firearm_exposure_level'].isin([3, 4]) & (df['firearm_and_protection'] == 3)).fillna(False).to_numpy()

# NULL if both columns are missing
missing = (df['firearm_exposure_level'].isna() & df['firearm_and_protection'].isna()).to_numpy()

df['gun_risk'] = np.select(
    [missing, high_risk],
    [np.nan,  1        ],
    default=0
)
print(get_auc(df))

                   Feature  ROC_SCORE
1                      age   0.798761
11                    lead   0.650825
10          blood_pressure   0.626034
3                education   0.597063
0                      sex   0.589845
17                diabetes   0.588884
7               loud_noise   0.570673
12                 cadmium   0.557254
8          very_loud_noise   0.555839
18                gun_risk   0.551012
9                      bmi   0.545036
5   firearm_and_protection   0.543493
4     income_poverty_ratio   0.538640
6   firearm_exposure_level   0.537902
2         military_service   0.534377
15                  stroke   0.524122
13           heart_disease   0.522675
16                  cancer   0.522260
14            heart_attack   0.516134


As you can see, my new column could perform a little better than each of those columns individually. Even though it sits at the borderline (0.551), I decide to keep it based on the severe effect of blast trauma on hearing loss. But for other columns below 0.55 and also hl_severity (as I don't need it now for EDA and it leakes data), I shall decide to "off with their heads!!".

In [64]:
cols_to_drop = [
    'bmi', 
    'firearm_and_protection', 
    'income_poverty_ratio', 
    'firearm_exposure_level', 
    'military_service', 
    'stroke', 
    'heart_disease',
    'cancer', 
    'heart_attack',
    'hl_severity'
]
df = df.drop(columns= cols_to_drop, errors = 'ignore')

Now moving forward to imputing missing values and outlier handling.

In [65]:
# NULL percentages
print(df.isnull().mean().sort_values(ascending= False) * 100)

cadmium            52.100255
lead               52.100255
blood_pressure      1.972615
gun_risk            0.116036
loud_noise          0.092829
very_loud_noise     0.069622
diabetes            0.046414
seqn                0.000000
education           0.000000
age                 0.000000
sex                 0.000000
hl_presence         0.000000
dtype: float64


Before I start for imputing, I check the outliers for lead and cadmium. Based on my research the maximum values recorded here are much lower than possible high values for these metals (>= 45–70 µg/dL for lead and >= 50 µg/L for cadmium can be seen in people who were highly exposured to industrial material).

For imputing, based on the fact that missing is because of sample design (missingness carries no value) and being heavily right skewed, we don't use missing flag and use median.

In [66]:
print(df.nlargest(10, 'lead')[['seqn','lead']])
print(df.nlargest(10, 'cadmium')[['seqn','cadmium']])


       seqn   lead
3487  91747  23.51
1862  87997  20.57
539   84919  19.58
387   84596  18.71
446   84738   8.34
2196  88755   7.95
3347  92569   7.86
448   84745   7.55
3743  92366   7.41
304   84423   7.24
       seqn  cadmium
1012  85995     9.17
1972  88233     6.37
1816  87892     5.14
3679  92202     4.94
2977  90571     3.78
3029  90673     3.70
1     83733     3.53
859   85663     3.47
150   84070     3.39
410   84652     3.35


In [67]:
df['lead'] = df['lead'].fillna(df['lead'].median())
df['cadmium'] = df['cadmium'].fillna(df['cadmium'].median())

In [68]:
# now I will go through the same steps for blood pressure
print(df.nlargest(10, 'blood_pressure')[['seqn','blood_pressure']])
print(df.nsmallest(10, 'blood_pressure')[['seqn', 'blood_pressure']])


       seqn  blood_pressure
1356  86748           232.0
465   84770           222.0
1200  86416           217.0
1002  85974           208.0
1115  86210           208.0
766   85429           206.0
4045  93049           205.0
3016  90649           202.0
2587  89623           199.0
1773  87783           198.0
       seqn  blood_pressure
2125  88607            76.0
845   85631            84.0
1596  87317            84.0
4239  93529            85.0
3252  91196            86.0
298   84410            87.0
947   85855            87.0
1586  87292            87.0
878   85706            88.0
2037  88400            88.0


These values are humanly possible and based on the slight skew in my distribution, I will use median for imputation. 

As for the rest of the columns with NULL values, because they all together carry less than 0.3% of all the data, I drop the rows where one of these columns are missing as well.


In [69]:
df['blood_pressure'] = df['blood_pressure'].fillna(df['blood_pressure'].median())

low_null_columns = ['gun_risk', 'loud_noise', 'very_loud_noise', 'diabetes']
before = len(df)
df = df.dropna(subset=low_null_columns)
after = len(df)
print(f"Dropped {before - after} rows which is {(before - after)/before * 100}")

Dropped 12 rows which is 0.2784868879090276


In [70]:
# Making sure there is no null left
print(df.isnull().sum().sum())
len(df)

0


4297

In [71]:
# Saving the file for the next phase
output_path = project_path / "data" / "processed" / "cleaned_master_table.parquet"
df.to_parquet(output_path, index=False)